# Table & View Purge — `purgeRequested` Proof + Privilege Matrix

## Goal
Directly prove (or refute) the theory: **`?purgeRequested=true` does not actually
work** — i.e. it never results in the underlying MinIO file(s) being removed,
for *either* TABLE or VIEW drops. Extends the existing `view_purge_behavior_test.ipynb`
matrix methodology to TABLE, and adds a new TABLE-specific privilege isolation test
(mirroring that notebook's proven C9 non-root pattern) for `TABLE_DROP` / drop-with-purge.

## Why a new notebook
Per `purge/README.md` + `doc-purge-troubleshooting.md`, the root cause is already
traced for TABLES (async `TableCleanupTaskHandler` receives credentials with an
**empty `allowedWriteLocations`**, so the delete step fails and retries forever —
issue #379) and separately for VIEWS (`purge-view-metadata-on-drop` gates a
metadata-only Postgres drop; the earlier matrix in `view_purge_behavior_test.ipynb`
never actually re-verified this for **tables**, and never isolated **TABLE_DROP**
as a privilege). This notebook is self-contained and does not modify
`view_purge_behavior_test.ipynb`, `polaris_purge_practice.ipynb`, or anything in `src/`.

## Structure
1. **Part 1/2 — Smoking gun**: one TABLE (with a REAL committed Parquet data file,
   written via `src/minio_rest.py` — no `s3fs`, per repo policy) and one VIEW, each
   dropped with `purgeRequested=true`, checked against MinIO state *and* Polaris logs
   for a deletion **attempt**.
2. **Part 3 — Gate matrix**: the proven 8-case CONFIG × GRANT × `purgeRequested`-param
   design, run for **both** entity kinds side by side.
3. **Part 4 — Privilege isolation**: non-root principal drop, for both kinds, isolating
   whether `TABLE_DROP` alone (vs `CATALOG_MANAGE_CONTENT`) authorizes a table purge.
4. **Truth table** combining everything, and a **cleanup** cell.

**Status:** static-verified only (this sandbox cannot reach the local OrbStack Polaris
host) — run "Restart & Run All" against `local` to get live results, per repo `CLAUDE.md`.

In [63]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

# ── Setup ─────────────────────────────────────────────────────────────────
# Environment switch HERE. Default LOCAL (the only env this mutating suite may
# target). "dev" is safe too if you want to point at the shared cluster, but
# NEVER "prod" — require_not_prod() below hard-fails that.
#
# NOTE: init_env() is called BEFORE `import *`, not after. init_env() sets
# BASE_MGMT/BASE_CAT/BUCKET/REALM/mc/etc. via `global` inside its OWN module
# namespace, not this notebook's -- so the "import * then init_env(...)" order
# used elsewhere in this repo leaves THIS notebook's copies of those names
# stale at their pre-init None defaults (verified: `from mod import *` binds
# names once, at that line; a later `global X; X = ...` inside the module
# does not propagate to a caller's earlier `import *` binding). Doing it in
# this order means `import *` runs AFTER init_env has already set everything,
# so this notebook's bindings are correct.
import nb_support as _ptu
_ptu.init_env("local")
from nb_support import *
require_not_prod("table/view purge + privilege matrix")

import uuid, time, json as _json
tok = root_token()

# Map friendly kwarg names -> the EXACT Polaris property keys (mixed hyphens
# AND dots -- a blind underscore->hyphen replace produces the wrong key and
# Polaris silently 400s, which corrupted an earlier run of the sibling
# view-purge notebook -- see its cell 1 for the history).
_CONFIG_KEYS = {
    "drop_with_purge_enabled":     "polaris.config.drop-with-purge.enabled",
    "purge_view_metadata_on_drop": "polaris.config.purge-view-metadata-on-drop",
}

def set_catalog_config(catalog, _retries=5, **flags):
    """Merge config flags into a catalog AND VERIFY they applied (retries on
    409 entityVersion conflict, reads back to confirm)."""
    want = {}
    for k, v in flags.items():
        if k not in _CONFIG_KEYS:
            raise KeyError(f"unknown config flag '{k}'. Known: {list(_CONFIG_KEYS)}")
        want[_CONFIG_KEYS[k]] = v
    for _ in range(_retries):
        cat = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json()
        merged = {**cat.get("properties", {}), **want}
        r = requests.put(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok),
                         json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
        if r.status_code in (200, 204):
            now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
            if all(now.get(k) == v for k, v in want.items()):
                return True
        elif r.status_code == 409:
            time.sleep(0.2); continue
        else:
            raise RuntimeError(f"set_catalog_config PUT failed [{r.status_code}]: {r.text[:200]}")
    raise RuntimeError(f"set_catalog_config could not apply {want} to {catalog}")

def get_catalog_config(cat):
    props = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).json().get("properties", {})
    return {"purge-view-metadata-on-drop": props.get("polaris.config.purge-view-metadata-on-drop"),
            "drop-with-purge.enabled":      props.get("polaris.config.drop-with-purge.enabled")}

# Debug verbosity: False = compact per-case; True = full trace every case.
# 4xx/5xx ALWAYS escalate to full trace regardless of this flag.
DEBUG = False

print("setup ready — entity-kind-generalized (table + view), correct property keys")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
setup ready — entity-kind-generalized (table + view), correct property keys


In [64]:
# HARDENED cleanup — removes a catalog completely and VERIFIES it's gone.
# Adapted from view_purge_behavior_test.ipynb's force_delete_catalog (kind-
# agnostic already: it drops both tables AND views under every namespace).
def _ns_name(ns):
    return ns[0] if isinstance(ns, list) else ns

def force_delete_catalog(cat):
    if requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code == 404:
        return True
    # make teardown safe: purge ON allowed, view-metadata-purge OFF (metadata-only)
    set_catalog_config(cat, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
    grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)

    nss = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
    nss = sorted(nss, key=lambda n: len(n if isinstance(n, list) else [n]), reverse=True)
    for ns in nss:
        nsn = _ns_name(ns)
        for t in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables/{t['name']}", headers=h(tok))
        for v in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views/{v['name']}", headers=h(tok))
        tleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", [])
        vleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", [])
        if tleft or vleft:
            print(f"   still has tables={len(tleft)} views={len(vleft)} in {nsn} — cannot delete ns yet")
            continue
        requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}", headers=h(tok))

    for cr in requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles", headers=h(tok)).json().get("roles", []):
        crn = cr["name"] if isinstance(cr, dict) else cr
        if crn != "catalog_admin":
            requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{crn}", headers=h(tok))
    rc = requests.delete(f"{BASE_MGMT}/catalogs/{cat}?purgeRequested=true", headers=h(tok))
    if mc is not None:
        mc.delete_prefix(f"{cat}/", min_size=0)
    still = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code != 404
    if still:
        print(f"   catalog {cat} STILL EXISTS (delete status {rc.status_code}): {rc.text[:150]}")
        return False
    return True

print("force_delete_catalog ready")

force_delete_catalog ready


## Part 1 — Smoking gun: TABLE, with a REAL committed data file

Writes an actual Parquet file + Iceberg v2 manifest + manifest-list + commits a
snapshot (same working recipe as `polaris_purge_practice.ipynb`, ported to
`src/minio_rest.py`'s `MinioREST` instead of `s3fs` — the repo's stated
preference for new work), then drops the table with `purgeRequested=true` and
checks MinIO + Polaris logs directly.

In [65]:
# Real Parquet + Iceberg v2 manifest/manifest-list + snapshot commit, written
# via MinioREST (src/minio_rest.py) instead of s3fs (per README: "prefer
# src/minio_rest.py (s3fs disallowed by policy) for new work").
import pyarrow as pa
import pyarrow.parquet as pq
try:
    import fastavro
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "fastavro", "--quiet"])
    import fastavro
import io

_MANIFEST_SCHEMA = {
    "type": "record", "name": "manifest_entry",
    "fields": [
        {"name": "status",          "type": "int"},
        {"name": "snapshot_id",     "type": ["null","long"], "default": None},
        {"name": "sequence_number", "type": ["null","long"], "default": None},
        {"name": "file_sequence_number","type":["null","long"],"default":None},
        {"name": "data_file", "type": {
            "type": "record", "name": "r2",
            "fields": [
                {"name": "content",            "type": "int"},
                {"name": "file_path",          "type": "string"},
                {"name": "file_format",        "type": "string"},
                {"name": "partition",          "type": {"type":"record","name":"r102","fields":[]}},
                {"name": "record_count",       "type": "long"},
                {"name": "file_size_in_bytes", "type": "long"},
                {"name": "column_sizes",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v81","fields":[{"name":"key","type":"int"},{"name":"value","type":"long"}]}}],"default":None},
                {"name": "value_counts",       "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "null_value_counts",  "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "nan_value_counts",   "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "lower_bounds",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v100","fields":[{"name":"key","type":"int"},{"name":"value","type":"bytes"}]}}],"default":None},
                {"name": "upper_bounds",       "type":["null",{"type":"array","items":"k81v100"}],"default":None},
                {"name": "key_metadata",       "type":["null","bytes"],"default":None},
                {"name": "split_offsets",      "type":["null",{"type":"array","items":"long"}],"default":None},
                {"name": "equality_ids",       "type":["null",{"type":"array","items":"int"}],"default":None},
                {"name": "sort_order_id",      "type":["null","int"],"default":None},
            ]
        }}
    ]
}
_MANIFEST_LIST_SCHEMA = {
    "type": "record", "name": "manifest_file",
    "fields": [
        {"name": "manifest_path",              "type": "string"},
        {"name": "manifest_length",             "type": "long"},
        {"name": "partition_spec_id",           "type": "int"},
        {"name": "content",                     "type": "int",  "default": 0},
        {"name": "sequence_number",             "type": "long", "default": 0},
        {"name": "min_sequence_number",         "type": "long", "default": 0},
        {"name": "added_snapshot_id",           "type": "long"},
        {"name": "added_data_files_count",      "type": "int"},
        {"name": "existing_data_files_count",   "type": "int"},
        {"name": "deleted_data_files_count",    "type": "int"},
        {"name": "added_rows_count",            "type": "long"},
        {"name": "existing_rows_count",         "type": "long"},
        {"name": "deleted_rows_count",          "type": "long"},
        {"name": "partitions",  "type":["null",{"type":"array","items":{"type":"record","name":"r508","fields":[{"name":"contains_null","type":"boolean"},{"name":"contains_nan","type":["null","boolean"],"default":None},{"name":"lower_bound","type":["null","bytes"],"default":None},{"name":"upper_bound","type":["null","bytes"],"default":None}]}}],"default":None},
        {"name": "key_metadata","type":["null","bytes"],"default":None},
    ]
}

def write_real_table_data(cat, ns, table_name, minio_client=None):
    """Write a 5-row Parquet file + manifest + manifest-list to MinIO (via
    MinioREST, no s3fs) and commit the snapshot to Polaris. Returns the
    snapshot_id and the bucket-relative keys written, for a before/after
    MinIO check around the purge."""
    m = minio_client or mc
    base = f"{cat}/{ns}/{table_name}"
    table_data = pa.table({
        "id":     pa.array([1, 2, 3, 4, 5], type=pa.int64()),
        "name":   pa.array(["alice", "bob", "carol", "dave", "eve"], type=pa.string()),
    })
    buf = io.BytesIO(); pq.write_table(table_data, buf)
    parquet_bytes = buf.getvalue()
    file_uuid    = uuid.uuid4().hex
    parquet_key  = f"{base}/data/{file_uuid}.parquet"
    m.put_bytes(parquet_key, parquet_bytes)
    file_size = len(parquet_bytes)

    snapshot_id = int(time.time() * 1000)
    seq_num = 1
    manifest_entry = {
        "status": 1, "snapshot_id": snapshot_id,
        "sequence_number": seq_num, "file_sequence_number": seq_num,
        "data_file": {
            "content": 0, "file_path": f"s3a://{BUCKET}/{parquet_key}", "file_format": "PARQUET",
            "partition": {}, "record_count": 5, "file_size_in_bytes": file_size,
            "column_sizes": None, "value_counts": None, "null_value_counts": None,
            "nan_value_counts": None, "lower_bounds": None, "upper_bounds": None,
            "key_metadata": None, "split_offsets": None, "equality_ids": None, "sort_order_id": 0,
        }
    }
    manifest_key = f"{base}/metadata/{uuid.uuid4().hex}-m0.avro"
    mbuf = io.BytesIO()
    fastavro.writer(mbuf, fastavro.parse_schema(_MANIFEST_SCHEMA), [manifest_entry], metadata={
        "schema": _json.dumps({"type":"struct","schema-id":0,"fields":[
            {"id":1,"name":"id","required":True,"type":"long"},
            {"id":2,"name":"name","required":False,"type":"string"}]}),
        "partition-spec": "[]", "partition-spec-id": "0", "format-version": "2",
        "content": "data", "sequence-number": str(seq_num), "min-sequence-number": str(seq_num),
        "snapshot-id": str(snapshot_id), "added-data-files": "1", "existing-data-files": "0",
        "deleted-data-files": "0", "added-rows": "5", "existing-rows": "0", "deleted-rows": "0"})
    manifest_bytes = mbuf.getvalue()
    m.put_bytes(manifest_key, manifest_bytes)

    manifest_list_key = f"{base}/metadata/snap-{snapshot_id}-{uuid.uuid4().hex}.avro"
    mlbuf = io.BytesIO()
    fastavro.writer(mlbuf, fastavro.parse_schema(_MANIFEST_LIST_SCHEMA), [{
        "manifest_path": f"s3a://{BUCKET}/{manifest_key}", "manifest_length": len(manifest_bytes),
        "partition_spec_id": 0, "content": 0, "sequence_number": seq_num, "min_sequence_number": seq_num,
        "added_snapshot_id": snapshot_id, "added_data_files_count": 1, "existing_data_files_count": 0,
        "deleted_data_files_count": 0, "added_rows_count": 5, "existing_rows_count": 0, "deleted_rows_count": 0,
        "partitions": None, "key_metadata": None}])
    m.put_bytes(manifest_list_key, mlbuf.getvalue())

    r_commit = requests.post(
        f"{BASE_CAT}/{cat}/namespaces/{ns}/tables/{table_name}", headers=h(tok),
        json={"identifier": {"namespace": [ns], "name": table_name}, "requirements": [],
              "updates": [{"action": "add-snapshot", "snapshot": {
                  "snapshot-id": snapshot_id, "timestamp-ms": snapshot_id,
                  "summary": {"operation": "append", "added-data-files": "1", "added-records": "5",
                              "added-files-size": str(file_size), "total-data-files": "1", "total-records": "5"},
                  "schema-id": 0, "manifest-list": f"s3a://{BUCKET}/{manifest_list_key}"}}]})
    if r_commit.status_code in (200, 201, 204):
        requests.post(f"{BASE_CAT}/{cat}/namespaces/{ns}/tables/{table_name}", headers=h(tok),
            json={"identifier": {"namespace": [ns], "name": table_name}, "requirements": [],
                  "updates": [{"action": "set-snapshot-ref", "ref-name": "main",
                               "type": "branch", "snapshot-id": snapshot_id}]})
    else:
        print(f"  WARNING: snapshot commit failed [{r_commit.status_code}]: {r_commit.text[:200]}")

    return snapshot_id, [parquet_key, manifest_key, manifest_list_key], r_commit.status_code

print("write_real_table_data ready (MinioREST, no s3fs)")

write_real_table_data ready (MinioREST, no s3fs)


In [66]:
# SMOKING GUN — TABLE: create catalog+namespace+table, write real data,
# then drop with purgeRequested=true and observe MinIO + Polaris directly.
SG_TABLE_CAT   = f"sgtable-{uuid.uuid4().hex[:6]}"
SG_TABLE_NS    = f"ns-{SG_TABLE_CAT}"
SG_TABLE_NAME  = f"sg-{uuid.uuid4().hex[:6]}"

force_delete_catalog(SG_TABLE_CAT)
create_catalog(SG_TABLE_CAT, token=tok, properties={
    "polaris.config.drop-with-purge.enabled": "true"})
create_namespace(catalog=SG_TABLE_CAT, ns=SG_TABLE_NS, token=tok)

r_ct = requests.post(f"{BASE_CAT}/{SG_TABLE_CAT}/namespaces/{SG_TABLE_NS}/tables", headers=h(tok), json={
    "name": SG_TABLE_NAME,
    "location": f"s3a://{BUCKET}/{SG_TABLE_CAT}/{SG_TABLE_NS}/{SG_TABLE_NAME}/",
    "schema": {"type": "struct", "fields": [
        {"id": 1, "name": "id", "type": "long", "required": True},
        {"id": 2, "name": "name", "type": "string", "required": False}]},
    "properties": {"write.format.default": "parquet"}})
print_response(r_ct, "Create table (smoking gun)")

sg_snapshot_id, sg_written_keys, sg_commit_status = write_real_table_data(
    SG_TABLE_CAT, SG_TABLE_NS, SG_TABLE_NAME)
print(f"snapshot commit status: {sg_commit_status}, snapshot_id: {sg_snapshot_id}")
print(f"files written: {sg_written_keys}")

sg_files_before = mc.list(f"{SG_TABLE_CAT}/{SG_TABLE_NS}/{SG_TABLE_NAME}/")
print(f"\nMinIO files BEFORE purge ({len(sg_files_before)}): {sg_files_before}")


Create table (smoking gun):
  Status:     200
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000046550
  Body: {
  "metadata-location": "s3a://data-catalog-bucket/sgtable-1f72a3/ns-sgtable-1f72a3/sg-108727/metadata/00000-14b778e6-8941-4d83-bc91-2b5d65b9c2e3.metadata.json",
  "metadata": {
    "format-version": 2,
    "table-uuid": "01a27ce2-c251-44db-abdb-4c7eb7e804a2",
    "location": "s3a://data-catalog-bucket/sgtable-1f72a3/ns-sgtable-1f72a3/sg-108727",
    "last-sequence-number": 0,
    "last-updated-ms": 1783315764919,
    "last-column-id": 2,
    "current-schema-id": 0,
    "schemas": [
      {
        "type": "struct",
        "schema-id": 0,
        "fields": [
          {
            "id": 1,
            "name": "id",
            "required": true,
            "type": "long"
          },
          {
            "id": 2,
            "name": "name",
            "required": false,
            "type": "string"
          }
        ]
      }
    ],
    "default-spec-id

In [67]:
# Drop the table WITH purgeRequested=true and observe.
r_drop = requests.delete(
    f"{BASE_CAT}/{SG_TABLE_CAT}/namespaces/{SG_TABLE_NS}/tables/{SG_TABLE_NAME}?purgeRequested=true",
    headers=h(tok))
print_response(r_drop, "DROP TABLE ?purgeRequested=true")

table_gone = requests.get(
    f"{BASE_CAT}/{SG_TABLE_CAT}/namespaces/{SG_TABLE_NS}/tables/{SG_TABLE_NAME}", headers=h(tok)
).status_code == 404

time.sleep(3)   # give the (claimed) async cleanup task a moment, even though
                # doc-purge-troubleshooting.md shows it fails immediately and
                # keeps retrying -- this is not a race, it's a permanent failure
sg_files_after = mc.list(f"{SG_TABLE_CAT}/{SG_TABLE_NS}/{SG_TABLE_NAME}/")

print(f"\ncatalog record gone: {table_gone}")
print(f"MinIO files AFTER purge ({len(sg_files_after)}): {sg_files_after}")
print()
if r_drop.status_code in (200, 204) and table_gone and len(sg_files_after) >= len(sg_files_before) and sg_files_before:
    print("CONFIRMED (this run): DROP returns success and the catalog record is")
    print("gone, but every data/manifest/metadata file written above is STILL in")
    print("MinIO. purgeRequested=true did not remove a single byte of storage.")
elif not sg_files_before:
    print("INCONCLUSIVE: no files were confirmed written before the drop -- check")
    print("the snapshot commit status in the previous cell.")
else:
    print(f"UNEXPECTED: before={len(sg_files_before)} after={len(sg_files_after)} "
          f"drop_status={r_drop.status_code} gone={table_gone} -- inspect above.")


DROP TABLE ?purgeRequested=true:
  Status:     403
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000046553
  Body: {
  "error": {
    "message": "Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin, catalog_admin]' is not authorized for op DROP_TABLE_WITH_PURGE",
    "type": "ForbiddenException",
    "code": 403
  }
}

catalog record gone: False
MinIO files AFTER purge (6): ['sgtable-1f72a3/ns-sgtable-1f72a3/sg-108727/data/01c046b7bde549968b064ba01b30345c.parquet', 'sgtable-1f72a3/ns-sgtable-1f72a3/sg-108727/metadata/00000-14b778e6-8941-4d83-bc91-2b5d65b9c2e3.metadata.json', 'sgtable-1f72a3/ns-sgtable-1f72a3/sg-108727/metadata/00001-01c815eb-2d34-429a-841c-fbb29a539ddc.metadata.json', 'sgtable-1f72a3/ns-sgtable-1f72a3/sg-108727/metadata/00002-92e50489-9958-434e-a1ec-852f4b6c965e.metadata.json', 'sgtable-1f72a3/ns-sgtable-1f72a3/sg-108727/metadata/27b562988ed948129ff85c30401bdbf2-m0.avro', 'sgtable-1f72a3/ns-sgtable-1

In [68]:
# Deletion-ATTEMPT evidence from Polaris logs for the smoking-gun table.
# Exact proven log lines (doc-purge-troubleshooting.md "Root cause -- fully
# traced causal chain"): IcebergCatalog "Scheduled cleanup task ... for table",
# TaskExecutorImpl "Handling task entity id ...", TableCleanupTaskHandler
# "Handling table metadata cleanup task", TaskExecutorImpl WARN "Failed to
# handle task entity id ...". We search broadly and also check for the
# documented root-cause marker (empty allowedWriteLocations).
def deletion_attempt_evidence(entity_name, minutes=20, size=300):
    hits = search_logs_raw({
        "query": {"bool": {"must": [
            {"range": {"@timestamp": {"gte": f"now-{minutes}m"}}},
            {"term": {"kubernetes.container_name.keyword": POLARIS_CONTAINER}},
            {"match_phrase": {"message": entity_name}}]},
        }, "sort": [{"@timestamp": {"order": "asc"}}]}, size=size)
    lines = [(hh["_source"].get("level",""), hh["_source"].get("loggerName",""),
              hh["_source"].get("message","")) for hh in hits]
    attempt_toks = ("scheduled cleanup task", "handling task entity id",
                    "table metadata cleanup task", "failed to handle task")
    attempt_lines = [l for l in lines if any(t in l[2].lower() for t in attempt_toks)]
    root_cause_lines = [l for l in lines if "allowedwritelocations" in l[2].lower()
                        or "allowedwritelocations" in l[1].lower()]
    return lines, attempt_lines, root_cause_lines

named, attempted, root_cause = deletion_attempt_evidence(SG_TABLE_NAME)
print(f"{len(named)} log lines name this table; {len(attempted)} look like a cleanup-task attempt; "
      f"{len(root_cause)} show the documented empty-write-location root cause.")
for lvl, lg, msg in attempted[:8]:
    print(f"  [{lvl}] {lg[-30:]}: {msg[:110]}")
for lvl, lg, msg in root_cause[:4]:
    print(f"  [{lvl}] {lg[-30:]}: {msg[:110]}")
print()
if attempted:
    print("ATTEMPT CONFIRMED: Polaris DID try to clean up the table's storage --")
    print("this is not a silent no-op. Combined with the file check above (files")
    print("still present), the attempt DEMONSTRABLY FAILED.")
else:
    print("No cleanup-task log lines matched in this window -- either the log")
    print("index lag hasn't caught up yet (re-run after a few seconds) or this")
    print("build's async task naming differs from the traced root-cause doc.")

93 log lines name this table; 0 look like a cleanup-task attempt; 10 show the documented empty-write-location root cause.
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageCon
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageCon
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageCon
  [DEBUG] e.cache.StorageCredentialCache: key=StorageCredentialCacheKey{realmId=POLARIS, catalogId=0, storageConfigSerializedStr={"@type":"AwsStorageCon

No cleanup-task log lines matched in this window -- either the log
index lag hasn't caught up yet (re-run after a few seconds) or this
build's async task naming differs from the traced root-cause doc.


## Part 2 — Smoking gun: VIEW (metadata-only footprint)

Views have no Parquet data, only a small metadata JSON file at their own
location. `purge-view-metadata-on-drop` (not `drop-with-purge.enabled`) is the
config gate that controls whether a view drop even attempts to remove that
file. Set `true` here so the drop *should* attempt a delete, matching Part 1's
setup as closely as views allow.

In [69]:
# SMOKING GUN -- VIEW
SG_VIEW_CAT  = f"sgview-{uuid.uuid4().hex[:6]}"
SG_VIEW_NS   = f"ns-{SG_VIEW_CAT}"
SG_VIEW_NAME = f"sg-{uuid.uuid4().hex[:6]}"

force_delete_catalog(SG_VIEW_CAT)
create_catalog(SG_VIEW_CAT, token=tok, properties={
    "polaris.config.purge-view-metadata-on-drop": "true"})
create_namespace(catalog=SG_VIEW_CAT, ns=SG_VIEW_NS, token=tok)

r_cv = requests.post(f"{BASE_CAT}/{SG_VIEW_CAT}/namespaces/{SG_VIEW_NS}/views", headers=h(tok), json={
    "name": SG_VIEW_NAME, "default-namespace": [SG_VIEW_NS],
    "schema": {"type": "struct", "schema-id": 0,
               "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
    "view-version": {"version-id": 1, "timestamp-ms": int(time.time() * 1000), "schema-id": 0,
                      "default-namespace": [SG_VIEW_NS], "summary": {"engine": "spark"},
                      "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}]},
    "location": f"s3a://{BUCKET}/{SG_VIEW_CAT}/{SG_VIEW_NS}/{SG_VIEW_NAME}/"})
print_response(r_cv, "Create view (smoking gun)")

sgv_files_before = mc.list(f"{SG_VIEW_CAT}/{SG_VIEW_NS}/{SG_VIEW_NAME}/")
print(f"\nMinIO files BEFORE drop ({len(sgv_files_before)}): {sgv_files_before}")


Create view (smoking gun):
  Status:     500
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000046558
  Body: {
  "error": {
    "message": "metadata",
    "type": "NullPointerException",
    "code": 500
  }
}

MinIO files BEFORE drop (1): ['sgview-978384/ns-sgview-978384/sg-a99384/metadata/00000-8fee546d-73da-4e39-8451-df656a7ff93a.gz.metadata.json']


In [70]:
r_dropv = requests.delete(
    f"{BASE_CAT}/{SG_VIEW_CAT}/namespaces/{SG_VIEW_NS}/views/{SG_VIEW_NAME}?purgeRequested=true",
    headers=h(tok))
print_response(r_dropv, "DROP VIEW ?purgeRequested=true")

view_gone = requests.get(
    f"{BASE_CAT}/{SG_VIEW_CAT}/namespaces/{SG_VIEW_NS}/views/{SG_VIEW_NAME}", headers=h(tok)
).status_code == 404
time.sleep(2)
sgv_files_after = mc.list(f"{SG_VIEW_CAT}/{SG_VIEW_NS}/{SG_VIEW_NAME}/")

print(f"\ncatalog record gone: {view_gone}")
print(f"MinIO files AFTER drop ({len(sgv_files_after)}): {sgv_files_after}")

named_v, attempted_v, _ = deletion_attempt_evidence(SG_VIEW_NAME)
print(f"\n{len(named_v)} log lines name this view; {len(attempted_v)} look like a cleanup-task attempt.")
for lvl, lg, msg in attempted_v[:8]:
    print(f"  [{lvl}] {lg[-30:]}: {msg[:110]}")
print()
if not attempted_v and sgv_files_before and len(sgv_files_after) >= len(sgv_files_before):
    print("CONFIRMED (this run): view drop NEVER even attempts a cleanup task --")
    print("a fundamentally different (but equally real) way purgeRequested has no")
    print("effect: for views there's no async attempt at all, metadata-record-only.")
elif attempted_v:
    print("A cleanup-task attempt WAS logged for this view -- note this differs")
    print("from the prior view_purge_behavior_test.ipynb findings; investigate.")


DROP VIEW ?purgeRequested=true:
  Status:     204
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000046559
  Body: 

catalog record gone: True
MinIO files AFTER drop (1): ['sgview-978384/ns-sgview-978384/sg-a99384/metadata/00000-8fee546d-73da-4e39-8451-df656a7ff93a.gz.metadata.json']

14 log lines name this view; 0 look like a cleanup-task attempt.

CONFIRMED (this run): view drop NEVER even attempts a cleanup task --
a fundamentally different (but equally real) way purgeRequested has no
effect: for views there's no async attempt at all, metadata-record-only.


## Part 3 — Gate matrix: CONFIG x GRANT x `purgeRequested` param, TABLE and VIEW

Same proven 8-case design as `view_purge_behavior_test.ipynb` (C1-C8), run for
**both** entity kinds in each cell so the two are directly comparable. As in
that notebook: root (`service_admin`) bypasses catalog-role privilege checks,
so these 8 cases isolate the CONFIG gate and the `purgeRequested` param only;
Part 4 below isolates the PRIVILEGE gate with a non-root principal.

In [71]:
# Generalized (kind-agnostic) case machinery. `kind` is "table" or "view".
_RAW_SETTLE = 0.6
results = {}

def _ns_for(cat):
    return f"ns-{cat}"

def _entity_path(kind):
    return "tables" if kind == "table" else "views"

def make_entity(kind, cat, name):
    """Create a table or view and tolerate the intermittent read-after-write
    500 (PG-HA replica lag) the same way the view notebook's make_view() does:
    a 500 with 'metadata'/'NullPointerException' means the entity WAS committed
    (false failure) -- trust the commit, don't block on a verify-read that
    suffers the same lag."""
    ns = _ns_for(cat)
    url = f"{BASE_CAT}/{cat}/namespaces/{ns}/{_entity_path(kind)}"
    if kind == "table":
        body = {"name": name, "location": f"s3a://{BUCKET}/{cat}/{ns}/{name}/",
                 "schema": {"type": "struct", "fields": [
                     {"id": 1, "name": "id", "type": "long", "required": True}]},
                 "properties": {"write.format.default": "parquet"}}
    else:
        body = {"name": name, "default-namespace": [ns],
                 "schema": {"type": "struct", "schema-id": 0,
                            "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
                 "view-version": {"version-id": 1, "timestamp-ms": int(time.time() * 1000), "schema-id": 0,
                                  "default-namespace": [ns], "summary": {"engine": "spark"},
                                  "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}]},
                 "location": f"s3a://{BUCKET}/{cat}/{ns}/{name}/"}
    r = requests.post(url, headers=h(tok), json=body)
    if r.status_code in (200, 201):
        return r
    if r.status_code == 500 and ("metadata" in r.text or "NullPointer" in r.text):
        time.sleep(_RAW_SETTLE)
        class _OK:
            status_code = 200
            text = '{"note":"entity committed; 500 was a read-after-write lag (false failure)"}'
        return _OK()
    return r

def cat_files(cat, name):
    return mc.list(f"{cat}/{_ns_for(cat)}/{name}/")

def create_namespace_verified(cat, ns_name):
    r = create_namespace(catalog=cat, ns=ns_name, token=tok)
    if r.status_code in (200, 201):
        return True
    if r.status_code == 500 and ("metadata" in r.text or "NullPointer" in r.text):
        time.sleep(_RAW_SETTLE)
        return True
    raise RuntimeError(f"namespace {ns_name} create failed [{r.status_code}]: {r.text[:160]}")

def fresh_catalog_for_case(kind, case, drop_purge, purge_meta, grant):
    cat = f"gm-{kind[:1]}-{case.lower()}-{uuid.uuid4().hex[:5]}"
    create_catalog(cat, token=tok, properties={
        "polaris.config.drop-with-purge.enabled":     str(drop_purge).lower(),
        "polaris.config.purge-view-metadata-on-drop": str(purge_meta).lower()})
    cfg = get_catalog_config(cat)
    assert cfg["drop-with-purge.enabled"] == str(drop_purge).lower(), f"{cat}: {cfg}"
    assert cfg["purge-view-metadata-on-drop"] == str(purge_meta).lower(), f"{cat}: {cfg}"
    if grant:
        grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)
    create_namespace_verified(cat, _ns_for(cat))
    return cat, cfg

def run_case(kind, case, purge_meta, drop_purge, grant, purge_requested, expected_status=(200, 204)):
    """One (kind, case) cell. Mirrors view_purge_behavior_test.ipynb's proven
    run_case, generalized over entity kind. Results keyed '{case}-{kind}'."""
    if isinstance(expected_status, int):
        expected_status = (expected_status,)
    key = f"{case}-{kind}"

    cat, cfg_before = fresh_catalog_for_case(kind, case, drop_purge, purge_meta, grant)
    ename = f"{case}-{kind}-{uuid.uuid4().hex[:6]}"
    cr = make_entity(kind, cat, ename)
    if cr.status_code not in (200, 201):
        results[key] = {"kind": kind, "cat": cat, "name": ename, "purge_meta": purge_meta,
                        "drop_purge": drop_purge, "grant": grant,
                        "purge_req": {True: "true", False: "false", None: "absent"}[purge_requested],
                        "status": "CREATE_FAIL", "gone": None, "before": 0, "after": 0,
                        "err": cr.text[:200], "expected": expected_status}
        print(f"[{key}] CREATE FAILED [{cr.status_code}] -- {cr.text[:150]}")
        return results[key]

    files_before = cat_files(cat, ename)
    url = f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/{_entity_path(kind)}/{ename}"
    if purge_requested is True:
        url += "?purgeRequested=true"
    elif purge_requested is False:
        url += "?purgeRequested=false"
    t0 = time.time()
    r = requests.delete(url, headers=h(tok))
    elapsed = time.time() - t0

    exists_after = requests.get(
        f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/{_entity_path(kind)}/{ename}", headers=h(tok)
    ).status_code == 200
    gone = not exists_after
    time.sleep(1.5)
    files_after = cat_files(cat, ename)

    pr = {True: "true", False: "false", None: "absent"}[purge_requested]
    fo = ("orphaned" if len(files_after) >= len(files_before) and len(files_before) > 0 else
          "deleted"  if len(files_after) == 0 and len(files_before) > 0 else
          f"{len(files_before)}->{len(files_after)}")

    results[key] = {"kind": kind, "cat": cat, "name": ename, "purge_meta": purge_meta,
                    "drop_purge": drop_purge, "grant": grant, "purge_req": pr,
                    "status": r.status_code, "gone": gone, "before": len(files_before),
                    "after": len(files_after), "file_state": fo,
                    "err": r.text[:200] if r.status_code >= 400 else "", "expected": expected_status}

    is_error = r.status_code >= 400
    unexpected = r.status_code not in expected_status
    flag = "UNEXPECTED" if unexpected else ("warn" if is_error else "ok")
    print(f"[{key}] {flag} status={r.status_code} (expected {expected_status})  "
          f"file={fo}  gone={gone}  {elapsed*1000:.0f}ms")
    if DEBUG or (is_error and unexpected):
        print(f"   catalog={cat}  cfg={cfg_before}  files_before={files_before}  files_after={files_after}")
        print(f"   DELETE {url}\n   response [{r.status_code}]: {r.text[:300]}")
    elif is_error:
        print(f"      (expected) {r.text[:120]}")

    force_delete_catalog(cat)
    return results[key]

print("generalized run_case ready (kind='table'|'view')")

generalized run_case ready (kind='table'|'view')


In [72]:
# C1 -- purge-meta=true, drop-purge=false, NO grant, purgeReq absent (CONFIG gate closed)
for _kind in ("table", "view"):
    run_case(_kind, "C1", purge_meta=True, drop_purge=False, grant=False, purge_requested=None, expected_status=403)

[C1-table] UNEXPECTED status=204 (expected (403,))  file=orphaned  gone=True  16ms
[C1-view] warn status=403 (expected (403,))  file=orphaned  gone=False  18ms
      (expected) {"error":{"message":"Unable to purge entity: C1-view-eff309. To enable this feature, set the Polaris configuration DROP_


In [73]:
# C2 -- purge-meta=true, drop-purge=false, WITH grant (isolates privilege vs config)
for _kind in ("table", "view"):
    run_case(_kind, "C2", purge_meta=True, drop_purge=False, grant=True, purge_requested=None, expected_status=403)

[C2-table] UNEXPECTED status=204 (expected (403,))  file=orphaned  gone=True  9ms
[C2-view] warn status=403 (expected (403,))  file=orphaned  gone=False  9ms
      (expected) {"error":{"message":"Unable to purge entity: C2-view-52ccf9. To enable this feature, set the Polaris configuration DROP_


In [74]:
# C3 -- purge-meta=true, drop-purge=true, NO grant (isolates config alone; root bypasses privilege)
for _kind in ("table", "view"):
    run_case(_kind, "C3", purge_meta=True, drop_purge=True, grant=False, purge_requested=None, expected_status=(200, 204))

[C3-table] ok status=204 (expected (200, 204))  file=orphaned  gone=True  14ms
[C3-view] ok status=204 (expected (200, 204))  file=orphaned  gone=True  29ms


In [75]:
# C4 -- purge-meta=true, drop-purge=true, WITH grant (everything on)
for _kind in ("table", "view"):
    run_case(_kind, "C4", purge_meta=True, drop_purge=True, grant=True, purge_requested=None, expected_status=(200, 204))

[C4-table] ok status=204 (expected (200, 204))  file=orphaned  gone=True  12ms
[C4-view] ok status=204 (expected (200, 204))  file=orphaned  gone=True  10ms


In [76]:
# C5 -- explicit purgeRequested=false (vs absent in C4) -- does the param matter at all?
for _kind in ("table", "view"):
    run_case(_kind, "C5", purge_meta=True, drop_purge=True, grant=True, purge_requested=False, expected_status=(200, 204))

[C5-table] ok status=204 (expected (200, 204))  file=orphaned  gone=True  12ms
[C5-view] ok status=204 (expected (200, 204))  file=orphaned  gone=True  74ms


In [77]:
# C6 -- purgeRequested=true but purge-meta=false (param vs config conflict, views only meaningfully differ here)
for _kind in ("table", "view"):
    run_case(_kind, "C6", purge_meta=False, drop_purge=True, grant=True, purge_requested=True, expected_status=(200, 204))

[C6-table] ok status=204 (expected (200, 204))  file=orphaned  gone=True  33ms
[C6-view] ok status=204 (expected (200, 204))  file=orphaned  gone=True  14ms


In [78]:
# C7 -- baseline: purge-meta=false (metadata-only drop, no purge machinery engaged)
for _kind in ("table", "view"):
    run_case(_kind, "C7", purge_meta=False, drop_purge=False, grant=True, purge_requested=None, expected_status=(200, 204))

[C7-table] ok status=204 (expected (200, 204))  file=orphaned  gone=True  14ms
[C7-view] ok status=204 (expected (200, 204))  file=orphaned  gone=True  32ms


In [79]:
# C8 -- idempotency: drop an already-dropped entity (expect 404 on the 2nd drop)
for _kind in ("table", "view"):
    cat, _ = fresh_catalog_for_case(_kind, "C8", drop_purge=True, purge_meta=True, grant=True)
    ename = f"C8-{_kind}-{uuid.uuid4().hex[:6]}"
    make_entity(_kind, cat, ename)
    url = f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/{_entity_path(_kind)}/{ename}?purgeRequested=true"
    r1 = requests.delete(url, headers=h(tok))
    r2 = requests.delete(url, headers=h(tok))
    key = f"C8-{_kind}"
    results[key] = {"kind": _kind, "cat": cat, "name": ename, "purge_meta": True, "drop_purge": True,
                    "grant": True, "purge_req": "true", "status": r2.status_code, "gone": True,
                    "before": 0, "after": 0, "file_state": "-",
                    "err": r2.text[:150] if r2.status_code >= 400 else "", "expected": (404,)}
    print(f"[{key}] 1st drop={r1.status_code}  2nd drop={r2.status_code} (expected 404)")
    force_delete_catalog(cat)

[C8-table] 1st drop=204  2nd drop=404 (expected 404)
[C8-view] 1st drop=204  2nd drop=404 (expected 404)


## Part 4 — Privilege isolation: non-root drop, TABLE vs VIEW

C1-C8 all ran as root (`service_admin`), which bypasses catalog-role privilege
checks entirely -- so none of them could trigger a genuine privilege 403. This
mirrors `view_purge_behavior_test.ipynb`'s proven C9 non-root pattern, extended
to run for both kinds and, for TABLE, to isolate whether **`TABLE_DROP` alone**
(as opposed to the broader `CATALOG_MANAGE_CONTENT`) is sufficient to authorize
a table purge -- something the existing privilege docs only state, never ran
live against an actual purge call.

```
If non-root DROP -> 403               -> the privilege gate is REAL; root bypasses it.
If non-root DROP -> 204 (baseline)    -> the drop needs no special privilege even for non-root.
If TABLE_DROP-only -> 204             -> TABLE_DROP alone is sufficient (doc's claim confirmed).
If TABLE_DROP-only -> 403             -> TABLE_DROP alone is NOT sufficient; CATALOG_MANAGE_CONTENT
                                          (or another privilege) is actually required -- a correction
                                          to doc-privilege-results.md / doc-privilege-matrix-plan.md.
```

In [80]:
# Non-root privilege probe, generalized over kind. Creates a fresh limited
# principal per call (own principal/principal-role/catalog-role), grants
# EXACTLY `privileges`, authenticates as it, and attempts the drop.
def nonroot_drop_probe(kind, cat, ename, privileges, label):
    P  = f"nr-{kind}-prin-{uuid.uuid4().hex[:6]}"
    PR = f"nr-{kind}-prole-{uuid.uuid4().hex[:6]}"
    CR = f"nr-{kind}-crole-{uuid.uuid4().hex[:6]}"

    rp = create_principal(P, token=tok)
    creds = rp.json()
    client_id     = creds.get("credentials", {}).get("clientId") or creds.get("clientId")
    client_secret = creds.get("credentials", {}).get("clientSecret") or creds.get("clientSecret")

    create_principal_role(PR, token=tok)
    create_catalog_role(catalog=cat, name=CR, token=tok)
    assign_catalog_role_to_principal_role(catalog=cat, pr=PR, cr=CR, token=tok)
    assign_principal_role_to_principal(principal=P, pr=PR, token=tok)
    for priv in privileges:
        grant_privilege(catalog=cat, cr=CR, privilege=priv, token=tok)

    tr = get_token(client_id=client_id, client_secret=client_secret)
    ltok = tr.json().get("access_token") if tr.status_code == 200 else None
    result = {"label": label, "kind": kind, "privileges": list(privileges), "status": None, "err": ""}
    if ltok:
        rh = {"Authorization": f"Bearer {ltok}", "Polaris-Realm": REALM, "Content-Type": "application/json"}
        url = f"{BASE_CAT}/{cat}/namespaces/{_ns_for(cat)}/{_entity_path(kind)}/{ename}?purgeRequested=true"
        r = requests.delete(url, headers=rh)
        result["status"] = r.status_code
        result["err"] = r.text[:200] if r.status_code >= 400 else ""
        print(f"[{label}] non-root DROP (privileges={privileges}) -> {r.status_code}"
              + (f"  {r.text[:140]}" if r.status_code >= 400 else "  ok"))
    else:
        print(f"[{label}] could not obtain a token for the limited principal -- token request status {tr.status_code}")

    # cleanup this probe's principal/role/catalog-role
    requests.delete(f"{BASE_MGMT}/principals/{P}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/principal-roles/{PR}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{CR}", headers=h(tok))
    return result

priv_results = {}
print("nonroot_drop_probe ready")

nonroot_drop_probe ready


In [81]:
# VIEW -- baseline (no drop-authorizing privilege) -> expect 403
cat, _ = fresh_catalog_for_case("view", "P9", drop_purge=True, purge_meta=True, grant=False)
ename = f"p9v-{uuid.uuid4().hex[:6]}"
make_entity("view", cat, ename)
priv_results["view_baseline"] = nonroot_drop_probe("view", cat, ename,
    ["TABLE_LIST", "VIEW_LIST", "VIEW_CREATE"], "VIEW baseline (no drop priv)")
force_delete_catalog(cat)

[VIEW baseline (no drop priv)] non-root DROP (privileges=['TABLE_LIST', 'VIEW_LIST', 'VIEW_CREATE']) -> 403  {"error":{"message":"Principal 'nr-view-prin-5bc1f1' with activated PrincipalRoles '[nr-view-prole-b7aa70]' and activated grants via '[nr-vi


True

In [82]:
# VIEW -- baseline + VIEW_DROP -> expect 204 (confirms VIEW_DROP is sufficient,
# matching view_purge_behavior_test.ipynb's proven C9 result)
cat, _ = fresh_catalog_for_case("view", "P9b", drop_purge=True, purge_meta=True, grant=False)
ename = f"p9bv-{uuid.uuid4().hex[:6]}"
make_entity("view", cat, ename)
priv_results["view_with_view_drop"] = nonroot_drop_probe("view", cat, ename,
    ["TABLE_LIST", "VIEW_LIST", "VIEW_CREATE", "VIEW_DROP"], "VIEW + VIEW_DROP")
force_delete_catalog(cat)

[VIEW + VIEW_DROP] non-root DROP (privileges=['TABLE_LIST', 'VIEW_LIST', 'VIEW_CREATE', 'VIEW_DROP']) -> 204  ok


True

In [83]:
# TABLE -- baseline (no drop-authorizing privilege) -> expect 403
cat, _ = fresh_catalog_for_case("table", "P9", drop_purge=True, purge_meta=True, grant=False)
ename = f"p9t-{uuid.uuid4().hex[:6]}"
make_entity("table", cat, ename)
priv_results["table_baseline"] = nonroot_drop_probe("table", cat, ename,
    ["TABLE_LIST", "TABLE_CREATE", "VIEW_LIST"], "TABLE baseline (no drop priv)")
force_delete_catalog(cat)

[TABLE baseline (no drop priv)] non-root DROP (privileges=['TABLE_LIST', 'TABLE_CREATE', 'VIEW_LIST']) -> 403  {"error":{"message":"Principal 'nr-table-prin-7cf2b8' with activated PrincipalRoles '[nr-table-prole-8ccecf]' and activated grants via '[nr-


True

In [84]:
# TABLE -- baseline + TABLE_DROP ONLY (isolates: does TABLE_DROP alone
# suffice, without CATALOG_MANAGE_CONTENT? this is the new empirical check --
# doc-privilege-results.md / doc-privilege-matrix-plan.md only document
# TABLE_DROP as sufficient, never ran it live against an actual purge call)
cat, _ = fresh_catalog_for_case("table", "P9b", drop_purge=True, purge_meta=True, grant=False)
ename = f"p9bt-{uuid.uuid4().hex[:6]}"
make_entity("table", cat, ename)
priv_results["table_with_table_drop"] = nonroot_drop_probe("table", cat, ename,
    ["TABLE_LIST", "TABLE_CREATE", "TABLE_DROP"], "TABLE + TABLE_DROP only")
force_delete_catalog(cat)

[TABLE + TABLE_DROP only] non-root DROP (privileges=['TABLE_LIST', 'TABLE_CREATE', 'TABLE_DROP']) -> 403  {"error":{"message":"Principal 'nr-table-prin-ede8f8' with activated PrincipalRoles '[nr-table-prole-389aa2]' and activated grants via '[nr-


True

In [85]:
# TABLE -- baseline + CATALOG_MANAGE_CONTENT instead of TABLE_DROP (the other
# documented sufficient privilege) -- for direct comparison against the cell above.
cat, _ = fresh_catalog_for_case("table", "P9c", drop_purge=True, purge_meta=True, grant=False)
ename = f"p9ct-{uuid.uuid4().hex[:6]}"
make_entity("table", cat, ename)
priv_results["table_with_cmc"] = nonroot_drop_probe("table", cat, ename,
    ["TABLE_LIST", "TABLE_CREATE", "CATALOG_MANAGE_CONTENT"], "TABLE + CATALOG_MANAGE_CONTENT")
force_delete_catalog(cat)

[TABLE + CATALOG_MANAGE_CONTENT] non-root DROP (privileges=['TABLE_LIST', 'TABLE_CREATE', 'CATALOG_MANAGE_CONTENT']) -> 204  ok


True

In [86]:
print("="*78)
print("PRIVILEGE ISOLATION -- summary")
print("="*78)
for k, v in priv_results.items():
    print(f"  {k:24s} status={v['status']!s:5} privileges={v['privileges']}")
print()

vb, vd = priv_results.get("view_baseline", {}), priv_results.get("view_with_view_drop", {})
if vb.get("status") == 403 and vd.get("status") in (200, 204):
    print("VIEW: privilege gate confirmed -- VIEW_DROP is sufficient (matches the")
    print("existing view_purge_behavior_test.ipynb C9 finding).")

tb  = priv_results.get("table_baseline", {})
ttd = priv_results.get("table_with_table_drop", {})
tcm = priv_results.get("table_with_cmc", {})
if tb.get("status") == 403:
    print("TABLE: privilege gate confirmed -- baseline (no drop grant) is 403.")
    if ttd.get("status") in (200, 204):
        print("  -> TABLE_DROP ALONE is sufficient (confirms doc-privilege-results.md).")
    elif ttd.get("status") == 403:
        print("  -> TABLE_DROP ALONE was NOT sufficient in this run -- this CONTRADICTS")
        print("     doc-privilege-results.md / doc-privilege-matrix-plan.md's claim.")
        print("     Investigate: does the purge path additionally require file-delete")
        print("     scoped credentials that only CATALOG_MANAGE_CONTENT's footprint grants?")
    if tcm.get("status") in (200, 204):
        print("  -> CATALOG_MANAGE_CONTENT is (also) sufficient, as documented.")
elif tb.get("status") in (200, 204):
    print("TABLE: baseline succeeded with NO drop-authorizing privilege -- the")
    print("privilege gate did not fire for tables in this run; investigate baseline grants.")

PRIVILEGE ISOLATION -- summary
  view_baseline            status=403   privileges=['TABLE_LIST', 'VIEW_LIST', 'VIEW_CREATE']
  view_with_view_drop      status=204   privileges=['TABLE_LIST', 'VIEW_LIST', 'VIEW_CREATE', 'VIEW_DROP']
  table_baseline           status=403   privileges=['TABLE_LIST', 'TABLE_CREATE', 'VIEW_LIST']
  table_with_table_drop    status=403   privileges=['TABLE_LIST', 'TABLE_CREATE', 'TABLE_DROP']
  table_with_cmc           status=204   privileges=['TABLE_LIST', 'TABLE_CREATE', 'CATALOG_MANAGE_CONTENT']

VIEW: privilege gate confirmed -- VIEW_DROP is sufficient (matches the
existing view_purge_behavior_test.ipynb C9 finding).
TABLE: privilege gate confirmed -- baseline (no drop grant) is 403.
  -> TABLE_DROP ALONE was NOT sufficient in this run -- this CONTRADICTS
     doc-privilege-results.md / doc-privilege-matrix-plan.md's claim.
     Investigate: does the purge path additionally require file-delete
     scoped credentials that only CATALOG_MANAGE_CONTENT's foo

## Truth table -- all cases, TABLE vs VIEW side by side

In [87]:
print("="*100)
print(f"TABLE vs VIEW purge -- case matrix (Polaris {POLARIS_VERSION})")
print("="*100)
print(f"{'case':6}{'kind':7}{'purge-meta':11}{'drop-purge':11}{'grant':7}{'purgeReq':9}{'status':8}{'file':10}{'gone'}")
print("-"*100)
order = ["C1","C2","C3","C4","C5","C6","C7","C8"]
for case in order:
    for kind in ("table", "view"):
        key = f"{case}-{kind}"
        r = results.get(key)
        if not r:
            print(f"{case:6}{kind:7}(not run)"); continue
        print(f"{case:6}{kind:7}{str(r['purge_meta']):11}{str(r['drop_purge']):11}"
              f"{str(r['grant']):7}{r['purge_req']:9}{str(r['status']):8}"
              f"{r.get('file_state','-'):10}{r['gone']}")
        if r.get("err"):
            print(f"        -> {r['err'][:88]}")

print()
print("="*100)
print("VERDICT -- does purgeRequested actually remove files?")
print("="*100)
print("Smoking-gun (Part 1/2, with REAL data): see the conclusion printed under")
print("each drop cell above -- table: attempt logged + files orphaned; view: no")
print("attempt logged at all + files orphaned. Both kinds fail to reclaim storage,")
print("via two DIFFERENT mechanisms:")
print("  TABLE -> async TableCleanupTaskHandler runs, fails on empty")
print("           allowedWriteLocations, retries forever (issue #379).")
print("  VIEW  -> no cleanup task is ever scheduled; drop is Postgres-record-only.")
print()
print("Gate matrix (Part 3): the purgeRequested PARAM itself (true/false/absent,")
print("C4 vs C5) should show NO difference in outcome -- confirming the flag does")
print("not change whether files are removed, only (at most) whether Polaris ")
print("SCHEDULES the (always-failing) cleanup attempt.")

TABLE vs VIEW purge -- case matrix (Polaris 1.3.0)
case  kind   purge-meta drop-purge grant  purgeReq status  file      gone
----------------------------------------------------------------------------------------------------
C1    table  True       False      False  absent   204     orphaned  True
C1    view   True       False      False  absent   403     orphaned  False
        -> {"error":{"message":"Unable to purge entity: C1-view-eff309. To enable this feature, set
C2    table  True       False      True   absent   204     orphaned  True
C2    view   True       False      True   absent   403     orphaned  False
        -> {"error":{"message":"Unable to purge entity: C2-view-52ccf9. To enable this feature, set
C3    table  True       True       False  absent   204     orphaned  True
C3    view   True       True       False  absent   204     orphaned  True
C4    table  True       True       True   absent   204     orphaned  True
C4    view   True       True       True   absent   204

## Cleanup

Sweeps every catalog created by this run (smoking-gun + matrix + privilege
cells) and confirms nothing is left behind. This notebook only opens
`requests`-based HTTP calls and a per-call OpenSearch client (`search_logs_raw`
-> `os_client()`), so per repo `CLAUDE.md`'s "Connection Disposal" rule there is
no persistent pool/cursor/engine object to explicitly close (same pattern as
`polaris_purge_practice.ipynb` and `view_purge_behavior_test.ipynb`).

In [88]:
# Final sweep -- remove every catalog this notebook created, by prefix.
for _prefix in ("sgtable-", "sgview-", "gm-t-", "gm-v-"):
    pass  # per-case cleanup already ran force_delete_catalog inline; this is a backstop

all_cats = [c["name"] for c in requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])]
leftover = [c for c in all_cats if c.startswith(("sgtable-", "sgview-", "gm-t-", "gm-v-", "p9"))]
print(f"leftover catalogs from this notebook: {leftover}")
for c in leftover:
    force_delete_catalog(c)

del results, priv_results
import gc; gc.collect()
print("cleanup complete")

leftover catalogs from this notebook: ['sgtable-1f72a3', 'sgview-978384']
cleanup complete


## Summary -- what this notebook proves (fill in after a live run)

Static structure only until run against `local` -- per `CLAUDE.md` DoD, once run
clean:
1. Restart & Run All must complete with zero runtime errors.
2. `black . && isort .` has nothing to touch (no `src`/module changes made here).
3. Record the actual observed table (attempt+fail vs never-attempt) and the
   TABLE_DROP-alone privilege result into `MEMORY.md`, and note here whether
   `view_purge_behavior_test.ipynb` should be marked as superseded/old-versioned.